# 03 - Trays, Modules, And Event Selection

IceTray gets its name from the `I3Tray`: a tray is a chain of modules.

Frames enter the tray, each module gets a chance to inspect or change the frame, and some modules can decide whether to keep or drop frames.


In [13]:
from pathlib import Path
import os

from icecube import icetray, dataclasses, dataio
from I3Tray import I3Tray

DATA_GCD = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_0101_78_503_GCD.i3.zst')
DATA_FILE = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_Subrun00000000_00000000.i3.zst')

print('GCD exists: ', DATA_GCD.exists())
print('Data exists:', DATA_FILE.exists())


GCD exists:  True
Data exists: True


## A function that counts hit DOMs

A module can be an ordinary Python function. IceTray will call the function once per frame.

The function below (1) looks for a pulse map, (2) counts how many DOMs have pulses, and (3) writes that number back into the frame as a new key called `TutorialHitDOMs`.


In [14]:
def find_pulse_key(frame):
    possible_keys = ['SplitInIcePulses', 'SplitInIceDSTPulses', 'SRTInIcePulses', 'InIcePulses', 'OfflinePulses']
    for key in possible_keys:
        if key in frame:
            return key
    return None

def add_hit_dom_count(frame):
    pulse_key = find_pulse_key(frame)
    if pulse_key is None:
        # Returning True means: keep this frame moving through the tray.
        return True

    pulse_map = dataclasses.I3RecoPulseSeriesMap.from_frame(frame, pulse_key)
    hit_doms = 0
    for omkey, pulses_on_one_dom in pulse_map:
        if len(pulses_on_one_dom) > 0:
            hit_doms += 1

    frame['TutorialHitDOMs'] = icetray.I3Int(hit_doms)
    return True

print('Defined add_hit_dom_count. It adds a new key called TutorialHitDOMs.')


Defined add_hit_dom_count. It adds a new key called TutorialHitDOMs.


## A function that selects events

This function returns `False` if the frame has less than 20 DOM hits, in which case IceTray drops that frame from the rest of the tray.

It's an example of making event selections.

We could in principle define any requirement, like a certain zenith angle range in one of the reconstruction keys, or a certain set of filter passes (more on that later), etc.


In [23]:
def keep_events_with_many_doms(frame, minimum_doms=20):
    if 'TutorialHitDOMs' not in frame:
        return False

    hit_doms = frame['TutorialHitDOMs'].value
    return hit_doms >= minimum_doms


## A small QFilterMask helper

`QFilterMask` stores named filter decisions.

A filter counts as passed when both `condition_passed` and `prescale_passed` are true.

We will use this later as an optional selection.


In [24]:
def filter_passed(frame, filter_name):
    if 'QFilterMask' not in frame:  # <-- check if QFilterMask even exists in this frame.
        return False
    if filter_name not in frame['QFilterMask']:  # <-- check if the filter we want exists in QFilterMask.
        return False

    result = frame['QFilterMask'][filter_name]
    return result.condition_passed and result.prescale_passed  # <-- return whether that filter is "passed" or not.

def keep_muon_filter(frame):
    return filter_passed(frame, 'MuonFilter_13')  # <-- this is the filter we want ('MuonFilter_13'). We can specify another if desired.


## Choose an output location

The selected `.i3.zst` file can be large, so we write it under `/data/user/<username>/IceTray_tutorial/`.

You get a larger data storage quota in `/data/user/` than in `/home/<username>.`


In [25]:
username = os.environ.get('USER', 'icecube_user')
output_dir = Path(f'/data/user/{username}/IceTray_tutorial')
output_dir.mkdir(parents=True, exist_ok=True)
selected_file = output_dir / 'selected_min20doms.i3.zst'

print('Selected events will be written to:')
print(selected_file)


Selected events will be written to:
/data/user/ireistr/IceTray_tutorial/selected_min20doms.i3.zst


## Build and run the tray

Read this cell from top to bottom.

The tray (1) reads frames, (2) counts hit DOMs in Physics frames, (3) drops events with fewer than 20 hit DOMs, and (4) writes the frames that remain.

We only execute 200 frames. You can crank this up to however many events live in the .i3 file.


In [26]:
tray = I3Tray()

tray.AddModule('I3Reader', 'reader', FilenameList=[str(DATA_GCD), str(DATA_FILE)])  # <-- (1)

tray.AddModule(add_hit_dom_count, 'add_hit_dom_count', Streams=[icetray.I3Frame.Physics])  # <-- (2)
tray.AddModule(keep_events_with_many_doms, 'minimum_dom_selection', minimum_doms=20, Streams=[icetray.I3Frame.Physics])  # <-- (3)

tray.AddModule(  # <-- (4)
    'I3Writer',
    'writer',
    Filename=str(selected_file),
    Streams=[icetray.I3Frame.TrayInfo, icetray.I3Frame.DAQ, icetray.I3Frame.Physics],
)

print('Starting the tray now...')
tray.Execute(200)
tray.Finish()
print('Finished. Output written to:', selected_file)


Starting the tray now...
Finished. Output written to: /data/user/ireistr/IceTray_tutorial/selected_min20doms.i3.zst


NOTICE (I3Tray): I3Tray finishing... (I3Tray.cxx:525 in void I3Tray::Execute(bool, unsigned int))


## Practice

1. Change `minimum_doms=20` to `minimum_doms=5`. Hover over the final output file written to your /data/user/ directory. It should be larger than the first one since our selection was less strict.
2. Add a new tray module: add `keep_muon_filter` to the tray after `add_hit_dom_count` so that the events must pass the `'MuonFilter13'` filter.
    1. (This filter sees if the event looks like a muon track or not.)
3. Add a print statement inside `add_hit_dom_count` for the first few events to read out the number of DOMs hit for each inspected event.
4. Write a new function that keeps events with total charge above a threshold (see notebook _02). Then add it do the tray segment (the cell above).
5. Read in the new .i3 file we've made (in your /data/user/ directory) and parse information from it using the tools in notebook _02.
   1. E.g., plot a histogram of the number of hit DOMs and see if the `'minimum_dom_selection'` tray worked.